In [3]:
"""
Prototype de code pour répondre à la question 9 du TP1
Vous avez juste à remplacer la chaine de caractères retournée de chaques opérations par du code assembleur.
Vous pouvez aussi ajouter de nouvelles opérations en suivant le même modèle
"""

import re


class ParserBob:
    def __init__(self, programme):
        self.programme = programme.split(' ')

    def push(self, commande):
        return f"""// push : {commande}
        @{commande} 
        D=A
        @SP
        A=M
        M=D
        @SP
        M=M+1
        """

    def label(self, commande):
        return f"""// label : {commande}
        {commande} 
        """

    def add(self, commande):
        return f"""// add : {commande}
        @SP
        A=M-1
        D=M
        A=A-1
        D=D+M
        @SP
        A=M
        M=D
        @SP
        M=M+1 
        """

    def equal(self, commande):
        return f"""// equal : {commande}
        @SP
        A=M-1
        D=M
        @SP 
        A=M-1
        A=A-1
        D=D-M
        @END{instruction}
        D;JEQ
        @SP
        A=M
        //pour lire l'adresse au sommet à SP
        M=0
        (END){instruction}
        @SP
        A=M
        M=1
        """
    
    

    def load(self, commande):
        return f"""// load : {commande}
        @SP
        A=M-1
        D=M
        @SP
        A=M-1
        A=M
        M=A
        """

    def store(self, commande):
        return f"""// store : {commande}
        code assembleur de {commande} 
        """

    def JEQ(self, commande):
        return f"""// jump : {commande}
        code assembleur de {commande} 
        """

    def pattern(self):
        return re.compile(r"""
        (?P<ADD>\+) | # sans commentaire
        (?P<EQUAL>=) | # sans commentaire
        (?P<LOAD><-) | # sans commentaire
        (?P<STORE>->) |# sans commentaire
        (?P<JUMP>JEQ|JNE) |# un jump
        (?P<LABEL>\([a-zA-Z]+\)) | # des strings parenthésés
        (?P<INT>[0-9]+|[a-zA-Z]+) # des entiers ou des variables ou des labels 
        """, re.X)

    def parse(self):
        self.asm = []
        pattern = self.pattern()
        for commande in self.programme:
            group = pattern.fullmatch(commande)
            if group is None:
                print('SyntaxError : ' + commande)
                exit()

            match group.lastgroup:
                case 'INT':
                    self.asm.append(self.push(commande))
                case 'LABEL':
                    self.asm.append(self.label(commande))
                case 'ADD':
                    self.asm.append(self.add(commande))
                case 'EQUAL':
                    self.asm.append(self.equal(commande))
                case 'STORE':
                    self.asm.append(self.store(commande))
                case 'LOAD':
                    self.asm.append(self.load(commande))
                case 'JUMP':
                    self.asm.append(self.JEQ(commande))
                case _:
                    print('Groupe oublié : ' + commande)
                    exit()
        return "\n".join(self.asm)


    def bootstrap(self):
        return """// Bootstrap
        @1
        D=A
        @SP
        M=D
        """


if __name__ == "__main__":
    programme = input('Prog :')
    bob = ParserBob(programme)
    asm = bob.parse()
    print(bob.bootstrap())
    print(asm)



Prog :3
// Bootstrap
        @1
        D=A
        @SP
        M=D
        
// push : 3
        @3 
        D=A
        @SP
        A=M
        M=D
        @SP
        M=M+1
        
